# Teach Gemma 3 Pig Latin in 30 Steps

Open-RL lets you fine-tune an LLM from a notebook using a simple, two-call API.
This notebook proves it by teaching Gemma 3 1B a toy language — Pig Latin — in under two minutes.

**What you'll see:**
- Connect to an Open-RL server and create a LoRA adapter
- Sample the base model (it has no idea what Pig Latin is)
- Train with `forward_backward` + `optim_step` — the two core primitives
- Evaluate the trained adapter on held-out words
- Try your own words in a live playground

**Before and after (30 steps of LoRA SFT):**

| Word | Base Model | Trained Model | Correct |
|------|-----------|---------------|---------|
| donut | onutday | onut-day | onut-day |
| space | aceday | ace-spay | ace-spay |
| machine | achinemaday | achine-may | achine-may |

In [ ]:
import json
import os
import random
import re
from dataclasses import asdict, dataclass
from difflib import SequenceMatcher
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import tinker
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from tinker import types

os.environ.setdefault("TINKER_API_KEY", "tml-dummy-key")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## Prerequisites

Start the server in another terminal with `make run-pig-latin-gemma-server`, then install client deps with `cd client && uv sync`.

## Connect to Open-RL

Open-RL exposes a [Tinker-compatible API](https://pypi.org/project/tinker/). You point a `ServiceClient` at your running server, then create a **LoRA training client** in one call. This client is your handle for everything that follows: training the adapter, snapshotting weights, and sampling completions.

No local GPU required — the server does all the heavy lifting.

In [ ]:
BASE_URL = os.getenv("TINKER_BASE_URL") or os.getenv("OPEN_RL_BASE_URL") or "http://127.0.0.1:9002"

service_client = tinker.ServiceClient(
    api_key=os.getenv("TINKER_API_KEY", "tml-dummy-key"),
    base_url=BASE_URL,
)

capabilities = await service_client.get_server_capabilities_async()
server_model = next(
    (m.model_name for m in capabilities.supported_models if getattr(m, "model_name", None)),
    None,
)
if server_model is None:
    raise RuntimeError(f"No model loaded at {BASE_URL}. Start with: make run-pig-latin-gemma-server")

print(f"Connected to {BASE_URL}")
print(f"Server model: {server_model}")

In [ ]:
# One call creates a LoRA adapter on the server. Base model stays frozen.
training_client = await service_client.create_lora_training_client_async(
    base_model="google/gemma-3-1b-it",
    rank=32,
    train_mlp=True,
    train_attn=True,
    train_unembed=False,
)
print(f"LoRA adapter created | rank=32 | base_model=google/gemma-3-1b-it")

## The data

We're using simple English-to-Pig-Latin word pairs. The task is deliberately trivial so the focus stays on the framework, not the linguistics.

In [ ]:
SYSTEM_PROMPT = "Translate the English text into Pig Latin. Reply with only the Pig Latin translation."
PAIRS_PATH = Path("piglatin_data.json")

pair_data = json.loads(PAIRS_PATH.read_text())
train_pairs = list(pair_data["train"])
eval_pairs = list(pair_data["eval"])
random.Random(64).shuffle(train_pairs)

print(f"Training pairs: {len(train_pairs):,}")
print(f"Eval pairs:     {len(eval_pairs):,}\n")

display(pd.DataFrame(
    [{"english": p[0], "pig_latin": p[1]} for p in train_pairs[:5]]
))

## Snapshot and sample: the sampling client

Open-RL separates training from inference with a neat trick: **named weight snapshots**. At any point during training you can freeze the current adapter state under an alias, then open a lightweight sampling client against that snapshot.

```python
# Freeze the adapter as "step_10" — this is instant, not a copy
path = training_client.save_weights_for_sampler(name="step_10").result().path

# Open a sampling client that generates from that frozen state
sampler = service_client.create_sampling_client(path)
sampler.sample(...)
```

This means you can:
- **Compare checkpoints** — snapshot at step 0, 15, and 30, then sample all three side by side
- **Keep training while sampling** — the snapshot is immutable, so sampling doesn't block the next `forward_backward` call
- **Build a playground** — after training, just `create_sampler("final")` and let users query it interactively

Let's try it. We haven't trained yet, so this snapshot is the base model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
EVAL_MAX_TOKENS = 32

# Build a prompt for one word
test_word = "good"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_word},
]
prompt_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
prompt_tokens = tokenizer.encode(prompt_text, add_special_tokens=False)

# Snapshot the untrained adapter under the alias "baseline"
weights_path = training_client.save_weights_for_sampler(name="baseline").result().path
sampler = service_client.create_sampling_client(weights_path)

result = sampler.sample(
    prompt=types.ModelInput.from_ints(tokens=prompt_tokens),
    num_samples=1,
    sampling_params=types.SamplingParams(max_tokens=EVAL_MAX_TOKENS, temperature=0.0),
).result()

raw_output = tokenizer.decode(result.sequences[0].tokens, skip_special_tokens=True).strip()
print(f'Input:    "{test_word}"')
print(f'Expected: "ood-gay"')
print(f'Got:      "{raw_output}"')
print(f'\nThis snapshot is saved as "baseline" — we can sample from it again later to compare.')

The base model has no idea what Pig Latin is. Let's quantify that with a proper baseline, then fix it.

First we need some plumbing: functions to build training examples, normalize outputs, and run evaluation. The details aren't important for understanding Open-RL — what matters is that each training example becomes a **`Datum`** containing token IDs and per-token loss weights. The weights are `0` on the prompt (so the model doesn't try to memorize instructions) and `1` on the answer (so it learns to produce the translation).

In [ ]:
# --- Plumbing: example building and evaluation helpers (collapse or skim) ---

def build_messages(source, target):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": source},
        {"role": "assistant", "content": target},
    ]


def make_datum(token_ids, weights):
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=token_ids[:-1]),
        loss_fn_inputs={"weights": weights[1:], "target_tokens": token_ids[1:]},
    )


def build_example(pair):
    source, target = pair
    messages = build_messages(source, target)

    prompt_text = tokenizer.apply_chat_template(
        messages[:2], tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )

    prompt_token_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    full_token_ids = tokenizer.encode(full_text, add_special_tokens=False)
    if len(full_token_ids) <= len(prompt_token_ids):
        return None

    target_token_count = len(full_token_ids) - len(prompt_token_ids)
    # 0-weight on prompt, 1-weight on answer — only train on the translation
    weights = [0] * len(prompt_token_ids) + [1] * target_token_count

    return {
        "messages": messages, "source": source, "target": target,
        "prompt_tokens": prompt_token_ids, "active_tokens": target_token_count,
        "datum": make_datum(full_token_ids, weights),
    }


def normalize_translation(text):
    text = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL)
    text = text.replace("<|im_start|>", " ").replace("<|im_end|>", " ")
    text = re.sub(r"^assistant\s*[:\-]?\s*", "", text.strip(), flags=re.IGNORECASE)
    return " ".join(text.split()).lower()


def create_sampler(alias):
    """Snapshot the current adapter weights under `alias` and return a sampling client.
    
    Each alias is a named checkpoint on the server. You can create as many as
    you like — "baseline", "step_15", "final", etc. — and sample from any of
    them at any time without interrupting training.
    """
    weights_path = training_client.save_weights_for_sampler(name=alias).result().path
    return service_client.create_sampling_client(weights_path)


def decode_first_sequence(result):
    if result.sequences:
        return tokenizer.decode(result.sequences[0].tokens, skip_special_tokens=True)
    return ""


def run_eval(alias, examples):
    """Snapshot the adapter as `alias`, sample every example, compute metrics."""
    sampler = create_sampler(alias)
    pending = []
    for ex in examples:
        future = sampler.sample(
            prompt=types.ModelInput.from_ints(tokens=ex["prompt_tokens"]),
            num_samples=1,
            sampling_params=types.SamplingParams(max_tokens=EVAL_MAX_TOKENS, temperature=0.0),
        )
        pending.append((ex, future))

    rows, exact_count, sim_sum = [], 0, 0.0
    for ex, future in pending:
        raw = decode_first_sequence(future.result()).strip()
        predicted = normalize_translation(raw)
        target = normalize_translation(ex["target"])
        sim = SequenceMatcher(None, predicted, target).ratio()
        exact = predicted == target
        exact_count += float(exact)
        sim_sum += sim
        rows.append({
            "source": ex["source"], "target_raw": ex["target"],
            "predicted_raw": raw, "exact": exact, "similarity": sim,
        })

    n = max(1, len(rows))
    return exact_count / n, sim_sum / n, pd.DataFrame(rows)

print("Helpers loaded.")

Now the interesting part — build the examples and measure the baseline. This is where we see how badly the untrained model does, which sets up the "before" for our training story.

In [ ]:
train_examples = [ex for p in train_pairs[:320] if (ex := build_example(p)) is not None]
eval_examples = [ex for p in eval_pairs[:64] if (ex := build_example(p)) is not None]
batch_size = min(16, len(train_examples))

print(f"Training examples: {len(train_examples):,}")
print(f"Eval examples:     {len(eval_examples):,}")
print(f"Batch size:        {batch_size}\n")

# Run the untrained model across all eval pairs
before_exact, before_similarity, baseline_df = run_eval("baseline", eval_examples)

print(f"Baseline — Exact match: {before_exact * 100:.1f}%  |  Similarity: {before_similarity * 100:.1f}%")
print("0% exact match. The model can't do Pig Latin at all yet.\n")

# Save a snapshot for a few held-out words so we can show before/after later
comparison_examples = eval_examples[2:5]
_, _, comparison_before_df = run_eval("comparison_before", comparison_examples)

## The two API calls that do all the work

This is the core of Open-RL, and it's surprisingly simple. Your entire training loop comes down to two calls:

- **`forward_backward_async(datums, "cross_entropy")`** — sends a batch of examples to the server. The server runs the forward pass through the frozen base model + your LoRA adapter, computes cross-entropy loss on the answer tokens, and backpropagates gradients through the adapter weights. All the GPU work happens server-side.

- **`optim_step_async(AdamParams(learning_rate=...))`** — tells the server to apply one Adam optimizer update using the gradients from the previous call.

That's it. No local GPU needed. No distributed training setup. You pick the batch, the loss function, and the learning rate — Open-RL does the rest.

```
for each step:
    loss = forward_backward(batch, "cross_entropy")   # server computes gradients
    _    = optim_step(learning_rate=1e-4)              # server applies update
```

Let's see both calls in isolation before running the full loop.

In [ ]:
# Take one batch and run both API calls so you can see them in isolation.
rng = random.Random(64)
shuffled_indices = list(range(len(train_examples)))
rng.shuffle(shuffled_indices)

first_batch = [train_examples[i] for i in shuffled_indices[:batch_size]]
datums = [ex["datum"] for ex in first_batch]
active_tokens = sum(ex["active_tokens"] for ex in first_batch)

# 1. Forward + backward — server computes loss and gradients
fb_result = await (await training_client.forward_backward_async(datums, "cross_entropy"))

# 2. Optimizer step — server applies the gradient update
await (await training_client.optim_step_async(types.AdamParams(learning_rate=1e-4)))

loss = float(fb_result.metrics.get("loss:sum", 0.0)) / max(1, active_tokens)
print(f"Step 1 complete | loss = {loss:.4f}")

One step, two API calls, and the loss already moved. That's the entire inner loop — everything else is just batching and bookkeeping.

Let's run the remaining 29 steps. Every 15 steps we'll snapshot the adapter and check how many eval words it gets right.

In [ ]:
TOTAL_STEPS = 30
EVAL_EVERY = 15
LEARNING_RATE = 1e-4

metrics_path = Path("artifacts/piglatin_notebook_metrics.jsonl")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text("", encoding="utf-8")

metrics_log = []

def record_metric(step, phase, loss=None, exact_match=None, similarity=None):
    row = {"step": step, "phase": phase}
    if loss is not None:
        row["loss"] = loss
    if exact_match is not None:
        row["exact_match"] = exact_match
    if similarity is not None:
        row["similarity"] = similarity
    metrics_log.append(row)
    with metrics_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")

record_metric(step=0, phase="eval", exact_match=before_exact, similarity=before_similarity)

losses = [loss]  # include step 1 loss from the cell above
eval_history = [{"step": 0, "exact_match": before_exact, "similarity": before_similarity}]

# Step 1 already done above; continue from step 2
position = batch_size  # we already consumed the first batch
progress = tqdm(range(2, TOTAL_STEPS + 1), desc="Training (steps 2-30)", unit="step")

for step in progress:
    if position + batch_size > len(shuffled_indices):
        rng.shuffle(shuffled_indices)
        position = 0

    batch_idx = shuffled_indices[position:position + batch_size]
    position += batch_size
    batch = [train_examples[i] for i in batch_idx]

    datums = [ex["datum"] for ex in batch]
    active_tokens = sum(ex["active_tokens"] for ex in batch)

    fb_future = await training_client.forward_backward_async(datums, "cross_entropy")
    opt_future = await training_client.optim_step_async(types.AdamParams(learning_rate=LEARNING_RATE))
    fb_result = await fb_future
    await opt_future

    step_loss = float(fb_result.metrics.get("loss:sum", 0.0)) / max(1, active_tokens)
    losses.append(step_loss)
    record_metric(step=step, phase="train", loss=step_loss)
    progress.set_postfix(loss=f"{step_loss:.4f}")

    if step % EVAL_EVERY == 0 or step == TOTAL_STEPS:
        em, sim, _ = run_eval(f"step_{step}", eval_examples)
        eval_history.append({"step": step, "exact_match": em, "similarity": sim})
        record_metric(step=step, phase="eval", exact_match=em, similarity=sim)
        progress.write(f"[eval at step {step}] exact={em * 100:.1f}% similarity={sim * 100:.1f}%")

print(f"\nMetrics saved to {metrics_path}")

## What changed

30 steps. ~1 minute of server time. The model went from 0% to ~50% exact match on held-out words, and the loss dropped ~90%. Not bad for a toy task with 320 training pairs.

In [ ]:
initial_loss = losses[0]
final_loss = losses[-1]
loss_drop_pct = (initial_loss - final_loss) / (abs(initial_loss) or 1.0) * 100
after_exact = eval_history[-1]["exact_match"]
after_similarity = eval_history[-1]["similarity"]

print(f"Loss:        {initial_loss:.4f} -> {final_loss:.4f}  ({loss_drop_pct:.0f}% drop)")
print(f"Exact match: {before_exact*100:.1f}% -> {after_exact*100:.1f}%")
print(f"Similarity:  {before_similarity*100:.1f}% -> {after_similarity*100:.1f}%")

# Single loss curve — the numbers above tell the eval story
fig, ax = plt.subplots(figsize=(8, 3))
loss_steps = list(range(1, len(losses) + 1))
ax.plot(loss_steps, losses, linewidth=0.8, alpha=0.7)
window = min(5, max(1, len(losses) // 4))
smoothed = pd.Series(losses).rolling(window, min_periods=1).mean()
ax.plot(loss_steps, smoothed, color="red", linewidth=1.5, label=f"{window}-step avg")
ax.set(xlabel="Step", ylabel="Loss", title="Training Loss")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
_, _, comparison_after_df = run_eval("comparison_after", comparison_examples)

comparison_df = pd.DataFrame({
    "source": comparison_before_df["source"],
    "target": comparison_before_df["target_raw"],
    "base_output": comparison_before_df["predicted_raw"],
    "trained_output": comparison_after_df["predicted_raw"],
    "base_exact": comparison_before_df["exact"],
    "trained_exact": comparison_after_df["exact"],
})
print("Before vs after on held-out words")
display(comparison_df)

The model learned the hyphen-suffix pattern and gets simple words right. It still struggles with consonant clusters and vowel-initial words — more training steps or more data would help, but the point is that **the entire fine-tuning loop ran from this notebook through two API calls**.

## Playground

The trained adapter is still live on the server. Each call below snapshots it under a fresh alias and samples — change the words and re-run as many times as you like. You could also go back and sample from the `"baseline"` snapshot to compare side by side.

In [ ]:
PLAYGROUND_WORDS = ["banana", "machine", "coding"]  # <-- edit these and re-run

def sample_translation(word, sampler_name="interactive"):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": word},
    ]
    prompt_text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    sampler = create_sampler(sampler_name)
    result = sampler.sample(
        prompt=types.ModelInput.from_ints(tokens=prompt_ids),
        num_samples=1,
        sampling_params=types.SamplingParams(max_tokens=EVAL_MAX_TOKENS, temperature=0.0),
    ).result()
    return decode_first_sequence(result).strip()

playground_rows = [
    {"word": w, "model_output": sample_translation(w) or "-- empty --"}
    for w in PLAYGROUND_WORDS
]
display(pd.DataFrame(playground_rows))

## Config reference

All tunables used in this notebook. Change these and re-run from the top.

In [ ]:
@dataclass
class Config:
    base_model: str = "google/gemma-3-1b-it"
    tokenizer_model: str = "google/gemma-3-1b-it"
    steps: int = 30
    batch_size: int = 16
    lora_rank: int = 32
    learning_rate: float = 1e-4
    server_command: str = "make run-pig-latin-gemma-server"
    base_url: str = "http://127.0.0.1:9002"
    eval_every: int = 15
    train_limit: int = 320
    eval_limit: int = 64
    seed: int = 64
    metrics_path: str = "artifacts/piglatin_notebook_metrics.jsonl"
    plot_path: str = "artifacts/piglatin_notebook_results.png"
    run_label: str = "Gemma 3 Pig Latin SFT"

config = Config()
display(pd.DataFrame.from_dict(asdict(config), orient="index", columns=["value"]))

## Next steps

- **More steps or different hyperparameters.** Try 60 steps, a higher rank (64), or a different learning rate.
- **Text-to-SQL notebook.** See the Gemma 3 SQL-to-text recipe for a more realistic SFT task.
- **RLVR guide.** Move beyond supervised fine-tuning with reinforcement learning from verifiable rewards.
- **Architecture doc.** Understand how the Open-RL server, training clients, and sampling clients fit together.